# Practical Searches in Azure AI Search

## Objective
Run and compare the four required search types against your index:
1. Vector Search
2. Hybrid Search (vector + keyword)
3. Semantic Search (`query_type=semantic`)
4. Semantic Hybrid Search (semantic + vector)

## 0) Setup and Configuration

This section loads environment variables and initializes the Azure AI Search client with Entra ID authentication (`DefaultAzureCredential`).

Expected `.env` variables:
- `AZURE_SEARCH_ENDPOINT`
- `AZURE_SEARCH_INDEX` (optional default provided)
- `AZURE_SEARCH_SEMANTIC_CONFIG` (optional default provided)

If the first connectivity check returns `403 Forbidden`, the identity is authenticated but does not have the required Azure AI Search data-plane role.

In [1]:
import os
from typing import Any, Dict, List

import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import QueryType, VectorizableTextQuery
from azure.core.exceptions import HttpResponseError

# Load local environment variables for endpoint/index configuration.
load_dotenv(override=True)

AZURE_SEARCH_ENDPOINT = os.getenv('AZURE_SEARCH_ENDPOINT', '').strip()
AZURE_SEARCH_INDEX = os.getenv(
    'AZURE_SEARCH_INDEX', 'rag-1776279755924').strip()
AZURE_SEARCH_SEMANTIC_CONFIG = os.getenv(
    'AZURE_SEARCH_SEMANTIC_CONFIG', 'default-configuration').strip()

if not AZURE_SEARCH_ENDPOINT:
    raise ValueError('Missing AZURE_SEARCH_ENDPOINT in .env')

# DefaultAzureCredential tries multiple auth methods (VS Code, CLI, browser, etc.).
credential = DefaultAzureCredential(
    exclude_interactive_browser_credential=False)
search_client = SearchClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    index_name=AZURE_SEARCH_INDEX,
    credential=credential,
)

print('Azure AI Search client initialized.')
print(f'Endpoint: {AZURE_SEARCH_ENDPOINT}')
print(f'Index: {AZURE_SEARCH_INDEX}')
print(f'Semantic config (expected): {AZURE_SEARCH_SEMANTIC_CONFIG}')

Azure AI Search client initialized.
Endpoint: https://04-ai-search-resource.search.windows.net
Index: rag-1776279755924
Semantic config (expected): default-configuration


In [16]:
# Quick connectivity check
probe = search_client.search(search_text='*', top=1, include_total_count=True)
print(
    f'Connectivity OK. Document count (service-reported): {probe.get_count()}')

Connectivity OK. Document count (service-reported): 8


## 1) Query and Helper Functions

In [31]:
QUERY_TEXT = 'Difference between vector search and hybrid search'
TOP_K = 5

# Keep select fields small so result tables stay readable.
SELECT_FIELDS = ['chunk_id', 'title', 'chunk', 'parent_id']


def normalize_result(result: Dict[str, Any], mode: str, rank: int) -> Dict[str, Any]:
    # Flatten each raw Search result into a compact row for comparison.
    chunk_text = (result.get('chunk') or '').replace('\n', ' ').strip()
    return {
        'mode': mode,
        'rank': rank,
        'title': result.get('title', ''),
        'chunk_id': result.get('chunk_id', ''),
        'score': result.get('@search.score'),
        'reranker_score': result.get('@search.reranker_score'),
        'preview': chunk_text[:180] + ('...' if len(chunk_text) > 180 else ''),
    }


def run_query(mode: str, **search_kwargs) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    results = search_client.search(
        top=TOP_K, select=SELECT_FIELDS, include_total_count=True, **search_kwargs)

    for rank, item in enumerate(results, start=1):
        rows.append(normalize_result(item, mode, rank))

    df = pd.DataFrame(rows)
    if not df.empty:
        visible_cols = ['rank', 'title', 'score', 'preview']
        # Show reranker score only when the current mode actually returns it.
        if df['reranker_score'].notna().any():
            visible_cols.insert(3, 'reranker_score')
        display(df[visible_cols])
    return df


def explain_top_result(df: pd.DataFrame, mode: str) -> None:
    # Print a short, human-readable summary for the top ranked item.
    if df.empty:
        print(f'{mode}: no results returned.')
        return

    top = df.iloc[0]
    title = str(top.get('title', '')).strip() or 'Untitled result'
    score = top.get('score')
    reranker = top.get('reranker_score')

    score_txt = f'{score:.4f}' if isinstance(score, (int, float)) else 'n/a'
    reranker_txt = f'{reranker:.4f}' if isinstance(
        reranker, (int, float)) else 'n/a'

    print(f'{mode} top result: {title}')
    print(f'Score: {score_txt}', end='')
    if reranker_txt != 'n/a':
        print(f' | Reranker score: {reranker_txt}')
    else:
        print()

## 2) Vector Search

Purpose: retrieve semantically similar chunks using vector similarity only.

In [27]:
# Vector-only retrieval: semantic similarity without keyword weighting.
vector_query = VectorizableTextQuery(
    text=QUERY_TEXT,
    k_nearest_neighbors=TOP_K,
    fields='text_vector',
)

df_vector = run_query(
    'Vector Search',
    search_text='*',
    vector_queries=[vector_query],
)
explain_top_result(df_vector, 'Vector Search')

,rank,title,score,preview
0,1,06_hybrid_search_playbook.html,0.857815,Hybrid Search Playbook Hybrid search c...
1,2,08_quick_faq.html,0.807682,Azure AI Search Quick FAQ Which embeddings...
2,3,01_rag_pipeline_overview.html,0.804206,RAG Pipeline Overview for Azure AI Search ...
3,4,04_vector_profile_and_algorithm.html,0.787667,Vector Profile and Algorithm Configuration ...
4,5,07_operations_and_monitoring.html,0.786500,Search Operations and Monitoring Stabl...


Vector Search top result: 06_hybrid_search_playbook.html
Score: 0.8578


### Explanation:

For this one, I relied only on vector similarity. It did a good job finding chunks with similar meaning, even when wording was different.
The trade-off is that it can miss exact term intent when I need strict keyword precision.

## 3) Hybrid Search (Vector + Keyword)

Purpose: combine lexical matching (`search_text`) with vector similarity for more robust retrieval.

In [28]:
# Hybrid retrieval: keyword match + vector similarity in one request.
hybrid_vector_query = VectorizableTextQuery(
    text=QUERY_TEXT,
    k_nearest_neighbors=TOP_K,
    fields='text_vector',
)

df_hybrid = run_query(
    'Hybrid Search',
    search_text=QUERY_TEXT,
    vector_queries=[hybrid_vector_query],
)
explain_top_result(df_hybrid, 'Hybrid Search')

,rank,title,score,preview
0,1,06_hybrid_search_playbook.html,0.033060,Hybrid Search Playbook Hybrid search c...
1,2,08_quick_faq.html,0.032522,Azure AI Search Quick FAQ Which embeddings...
2,3,01_rag_pipeline_overview.html,0.032002,RAG Pipeline Overview for Azure AI Search ...
3,4,04_vector_profile_and_algorithm.html,0.031498,Vector Profile and Algorithm Configuration ...
4,5,07_operations_and_monitoring.html,0.030550,Search Operations and Monitoring Stabl...


Hybrid Search top result: 06_hybrid_search_playbook.html
Score: 0.0331


### Explanation:

Here I combined keyword matching with vector similarity, and the results felt more balanced.
In my case, this mode usually kept relevant exact terms while still catching semantically related chunks.

## 4) Semantic Search

Purpose: rank keyword results with semantic understanding using your semantic configuration.

In [29]:
# Semantic ranking over lexical results using the configured semantic profile.
try:
    df_semantic = run_query(
        'Semantic Search',
        search_text=QUERY_TEXT,
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name=AZURE_SEARCH_SEMANTIC_CONFIG,
        query_caption='extractive',
        query_answer='extractive|count-1',
    )
    explain_top_result(df_semantic, 'Semantic Search')
except HttpResponseError as ex:
    print('Semantic search failed. Verify semantic configuration name and index settings.')
    print(ex)
    df_semantic = pd.DataFrame()

,rank,title,score,reranker_score,preview
0,1,06_hybrid_search_playbook.html,2.195968,2.971246,Hybrid Search Playbook Hybrid search c...
1,2,08_quick_faq.html,1.706504,2.647969,Azure AI Search Quick FAQ Which embeddings...
2,3,01_rag_pipeline_overview.html,1.509178,2.521070,RAG Pipeline Overview for Azure AI Search ...
3,4,03_semantic_configuration_notes.html,3.491090,2.124286,Semantic Configuration Notes Semantic ...
4,5,04_vector_profile_and_algorithm.html,0.941617,1.937259,Vector Profile and Algorithm Configuration ...


Semantic Search top result: 06_hybrid_search_playbook.html
Score: 2.1960 | Reranker score: 2.9712


### Explanation:

This search starts from lexical matches and then uses semantic reranking to reorder them.
I found the top results easier to read and closer to what I would actually use in an answer.

## 5) Semantic Hybrid Search (Semantic + Vector)

Purpose: combine semantic reranking with vector retrieval for context-aware and semantically aligned top results.

In [32]:
# Semantic hybrid: vector retrieval followed by semantic reranking.
semantic_hybrid_vector_query = VectorizableTextQuery(
    text=QUERY_TEXT,
    k_nearest_neighbors=TOP_K,
    fields='text_vector',
)

try:
    df_semantic_hybrid = run_query(
        'Semantic Hybrid Search',
        search_text=QUERY_TEXT,
        vector_queries=[semantic_hybrid_vector_query],
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name=AZURE_SEARCH_SEMANTIC_CONFIG,
        query_caption='extractive',
        query_answer='extractive|count-1',
    )
    explain_top_result(df_semantic_hybrid, 'Semantic Hybrid Search')
except HttpResponseError as ex:
    print('Semantic hybrid search failed. Verify semantic configuration name and index settings.')
    print(ex)
    df_semantic_hybrid = pd.DataFrame()

,rank,title,score,reranker_score,preview
0,1,06_hybrid_search_playbook.html,0.033060,2.971246,Hybrid Search Playbook Hybrid search c...
1,2,08_quick_faq.html,0.032522,2.647969,Azure AI Search Quick FAQ Which embeddings...
2,3,01_rag_pipeline_overview.html,0.032002,2.521070,RAG Pipeline Overview for Azure AI Search ...
3,4,03_semantic_configuration_notes.html,0.016667,2.124286,Semantic Configuration Notes Semantic ...
4,5,04_vector_profile_and_algorithm.html,0.031498,1.937259,Vector Profile and Algorithm Configuration ...


Semantic Hybrid Search top result: 06_hybrid_search_playbook.html
Score: 0.0331 | Reranker score: 2.9712


### Explanation:

This was the strongest mode overall in my runs: vector retrieval gave broad semantic coverage, and semantic reranking refined the final order.
For a RAG pipeline, this is the one I would prioritize when I want both relevance and readable top results.